In [14]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
import pickle
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever,ContextualCompressionRetriever
from sentence_transformers import CrossEncoder
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [2]:
load_dotenv()

embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [3]:

vectorstore = Chroma(
    collection_name="Neural",
    embedding_function=embedding,
    persist_directory='../chroma.db/'
)

In [4]:
retriver = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={
        "k": 3,
        "fetch_k": 10
    }
)

In [5]:
with open("../processed/chunks.pkl", "rb") as f:
    splited_document = pickle.load(f)

In [6]:
keyword_retriver = BM25Retriever.from_documents(
    documents=splited_document,
)
keyword_retriver.k = 3

In [7]:
hybrid_retriver = EnsembleRetriever(
    retrievers=[retriver,keyword_retriver],
    weights=[0.7,0.3]
)

In [12]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
test_query = "What is Neural Network"
candidates = hybrid_retriver.invoke(test_query)
pairs = [
    (test_query, doc.page_content)
    for doc in candidates
]

scores = reranker.predict(pairs)
ranked_results = sorted(
    zip(scores,candidates),
    key=lambda x : x[0],
    reverse=True
)

for i, (score, doc) in enumerate(ranked_results[:3], 1):
    print(f"\n--- Result {i} ---")
    print(f"Relevance Score: {score:.4f}")
    print(doc.page_content)
    print("Metadata:", doc.metadata)

c:\Users\pc\Desktop\DocuRAG Universal Document Intelligence Assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pc\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 124


--- Result 1 ---
Relevance Score: 10.9822
NEURAL NETWORKS — COMPLETE INTRODUCTION

1. What is a Neural Network?

A neural network is a machine learning model inspired by the structure of the human brain. It consists of interconnected computational units called neurons. Neural networks learn patterns from data by adjusting numerical parameters called weights and biases.
Metadata: {'source': '../data/Neural_networks.txt'}

--- Result 2 ---
Relevance Score: 7.3640
Neural networks are widely used in image classification, speech recognition, natural language processing, recommendation systems, fraud detection, medical image analysis, and autonomous systems.

A basic neural network contains three types of layers: an input layer, one or more hidden layers, and an output layer.

The input layer receives the input features. Hidden layers transform the input using mathematical operations and activation functions. The output layer produces the final prediction.
Metadata: {'source': '../data/Neur

In [23]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature = 0
)
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=hybrid_retriver
)

In [24]:
test_query = "What is Neural Network?"

results = compression_retriever.invoke(test_query)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)

c:\Users\pc\Desktop\DocuRAG Universal Document Intelligence Assistant\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\pc\Desktop\DocuRAG Universal Document Intelligence Assistant\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\pc\Desktop\DocuRAG Universal Document Intelligence Assistant\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\pc\Desktop\DocuRAG Universal Document Intelligence Assistant\.venv\Lib\site-packages\lan


--- Result 1 ---
A neural network is a machine learning model inspired by the structure of the human brain. It consists of interconnected computational units called neurons. Neural networks learn patterns from data by adjusting numerical parameters called weights and biases.
Metadata: {'source': '../data/Neural_networks.txt'}

--- Result 2 ---
A basic neural network contains three types of layers: an input layer, one or more hidden layers, and an output layer.
Metadata: {'source': '../data/Neural_networks.txt'}


In [8]:
test_query = "What is Neural Network"
result = retriver.invoke(test_query)
for i , doc in enumerate(result,1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)


--- Result 1 ---
NEURAL NETWORKS — COMPLETE INTRODUCTION

1. What is a Neural Network?

A neural network is a machine learning model inspired by the structure of the human brain. It consists of interconnected computational units called neurons. Neural networks learn patterns from data by adjusting numerical parameters called weights and biases.
Metadata: {'source': '../data/Neural_networks.txt'}

--- Result 2 ---
Neural networks are widely used in image classification, speech recognition, natural language processing, recommendation systems, fraud detection, medical image analysis, and autonomous systems.

A basic neural network contains three types of layers: an input layer, one or more hidden layers, and an output layer.

The input layer receives the input features. Hidden layers transform the input using mathematical operations and activation functions. The output layer produces the final prediction.
Metadata: {'source': '../data/Neural_networks.txt'}

--- Result 3 ---
For example, 

In [9]:
test_query = "What is Neural Network"
result = keyword_retriver.invoke(test_query)
for i , doc in enumerate(result,1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)


--- Result 1 ---
NEURAL NETWORKS — COMPLETE INTRODUCTION

1. What is a Neural Network?

A neural network is a machine learning model inspired by the structure of the human brain. It consists of interconnected computational units called neurons. Neural networks learn patterns from data by adjusting numerical parameters called weights and biases.
Metadata: {'source': '../data/Neural_networks.txt'}

--- Result 2 ---
The basic update rule is:

new_weight = old_weight - learning_rate * gradient

The learning rate controls the size of each update.

If the learning rate is too large, training may become unstable. If it is too small, training can become extremely slow.

Common optimization algorithms include Stochastic Gradient Descent (SGD), Adam, and AdamW.

8. Training a Neural Network

Training involves repeatedly presenting data to the neural network and updating its parameters.
Metadata: {'source': '../data/Neural_networks.txt'}

--- Result 3 ---
21. Advantages of Neural Networks

Neura

In [10]:
test_query = "What is Neural Network"
result = hybrid_retriver.invoke(test_query)
for i , doc in enumerate(result,1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)


--- Result 1 ---
NEURAL NETWORKS — COMPLETE INTRODUCTION

1. What is a Neural Network?

A neural network is a machine learning model inspired by the structure of the human brain. It consists of interconnected computational units called neurons. Neural networks learn patterns from data by adjusting numerical parameters called weights and biases.
Metadata: {'source': '../data/Neural_networks.txt'}

--- Result 2 ---
Neural networks are widely used in image classification, speech recognition, natural language processing, recommendation systems, fraud detection, medical image analysis, and autonomous systems.

A basic neural network contains three types of layers: an input layer, one or more hidden layers, and an output layer.

The input layer receives the input features. Hidden layers transform the input using mathematical operations and activation functions. The output layer produces the final prediction.
Metadata: {'source': '../data/Neural_networks.txt'}

--- Result 3 ---
For example, 

In [11]:
print("Total chunks:", len(splited_document))
print("Retrieved:", len(result))

Total chunks: 27
Retrieved: 5
